# YouTube Audience Engagement Analytics

# Business Problem

In [ ]:
#Businesses and content creators need to understand which YouTube content categories generate the highest audience engagement so they can create more effective content and improve viewer interaction.

# Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
!pip install plotly

In [ ]:
import plotly
print(plotly.__version__)

In [ ]:
# express is a sub module in plotly

In [ ]:
import plotly.express as px

# Data Collection and Data Understanding

In [ ]:
comments = pd.read_csv(r"D:\Sandeep Kaur\Projects\Youtube Analysis\YouTube_DataAnalysis_Project/UScomments.csv" , on_bad_lines="skip")

In [ ]:
comments

# Data Cleaning

In [ ]:
#False means not duplicate and True means duplicte rows

In [ ]:
# comments.duplicated() will return the boolean seriers, Identify duplicate rows and does not return the duplicate rows, returns only True/False values. 

In [ ]:
comments.duplicated()

In [ ]:
# comments[comments.duplicated()] => uses the Boolean Series to filter the DataFrame and return only the duplicate rows.

In [ ]:
# duplicated(), in comments[comments.duplicated()] => uses keep='first' by default, first occurrence is not marked/shown as a duplicate but the second (and later) occurrences are returned.

In [ ]:
# keep=False) => all rows involved in duplication, including the first occurrence.

In [ ]:
comments[comments.duplicated(keep=False)].sort_values("comment_text")

In [ ]:
comments = comments.drop_duplicates()

In [ ]:
# False => not a missing value AND True => missing values

In [ ]:
comments.isnull()

In [ ]:
# in comment_text => there are a few missing values => need to drop those M.V

In [ ]:
comments.isnull().sum()

In [ ]:
# drop na = not available values AND inplace = True => just to update dataframe

In [ ]:
comments.dropna(inplace = True)

In [ ]:
comments.isnull().sum()

# Sentiment Analysis

In [ ]:
!pip install nltk

In [ ]:
import nltk

In [ ]:
#  lexicon => builin list of words having a sentiment scores

In [ ]:
nltk.download("vader_lexicon")

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
sia = SentimentIntensityAnalyzer()

In [ ]:
comments["comment_text"]

In [ ]:
sia.polarity_scores("MY FAN . attendance")['compound']

In [ ]:
#using these values lexicon will give you sentiment score

In [ ]:
sia.lexicon

In [ ]:
def analyze_comment_sentiment(comment):
    compound = sia.polarity_scores(comment)['compound']
    
    if compound > 0.05:
        label = "Positive"
        insight = "The comment express positive emotion or approval"
        
    elif compound < 0.05:
        label = "Negative"
        insight = "The comment shows negative emotion or dis-satisfaction"

    else : 
        label = "Neutral"
        insight = "The comment is informational or emotionally neutral"
        
    return {
        "label": label,
        "score": compound,
        "insight": insight
    }

In [ ]:
analyze_comment_sentiment("MY FAN . attendance")

# Emoji Analysis

In [ ]:
!pip install emoji==2.15.0

In [ ]:
import emoji

In [ ]:
comments["comment_text"].head(25)

In [ ]:
comment = "trending 😉"

In [ ]:
emoji.EMOJI_DATA

In [ ]:
emoji_list = []
for char in comment:
    if char in emoji.EMOJI_DATA:
        emoji_list.append(char)

In [ ]:
emoji_list

In [ ]:
all_emoji_list = []

for comment in comments["comment_text"].dropna():
    for char in comment:
        if char in emoji.EMOJI_DATA:
            all_emoji_list.append(char)


In [ ]:
all_emoji_list[0:10]

In [ ]:
len(all_emoji_list)

In [ ]:
from collections import Counter
emoji_count_list = Counter(all_emoji_list).most_common(10)

In [ ]:
emoji_count_list

In [ ]:
 emojis = [emoji for emoji,count in emoji_count_list]

In [ ]:
 counts = [count for emoji,count in emoji_count_list]

In [ ]:
px.bar(x = emojis,
       y = counts,
       title = "Most used emojis in YouTube comments",
       labels = {
           "x" : "Emoji",
           "y" : "Count"
       })

In [ ]:
######     Shows that majority of youtube users are quite happy, 

In [ ]:
####         COLLECT ENTIRE DATA OF YOUTUBE

In [ ]:
import os

In [ ]:
files = os.listdir(r"D:\Sandeep Kaur\Projects\Youtube Analysis\YouTube_DataAnalysis_Project\additional_data")

In [ ]:
files_csv = [file for file in files if '.csv' in file]

In [ ]:
files_csv 

In [ ]:
full_df = pd.DataFrame()

path = r"D:\Sandeep Kaur\Projects\Youtube Analysis\YouTube_DataAnalysis_Project\additional_data"

for file in files_csv:
    current_df = pd.read_csv(path+'/'+file, encoding = "iso=8859-1", on_bad_lines="skip")
    full_df = pd.concat([current_df, full_df], ignore_index = True)

In [ ]:
full_df.shape

In [ ]:
full_df.head(4)

In [ ]:
full_df.columns

In [ ]:
### HOW TO EXPORT YOU DATA INTO (CSV,JSON, db)

In [ ]:
full_df[full_df.duplicated()].shape

In [ ]:
full_df = full_df.drop_duplicates()

In [ ]:
full_df.shape

In [ ]:
########  EXPORT DATA IN CSV FILE

In [ ]:
full_df[0:5000].to_csv(r"D:\Sandeep Kaur\Projects\Youtube Analysis\YouTube_DataAnalysis_Project\export_data/YouTube_Whole_Data.csv", index = "False")

In [ ]:
########  EXPORT DATA IN json FILE

In [ ]:
full_df[0:5000].to_json(r"D:\Sandeep Kaur\Projects\Youtube Analysis\YouTube_DataAnalysis_Project\export_data/YouTube_sample.json")

In [ ]:
########  EXPORT DATA IN DB FILE

In [ ]:
!pip install sqlalchemy

In [ ]:
#########   create_engine allows to connect to db

In [ ]:
from sqlalchemy import create_engine

In [ ]:
##    create_engine() will help to create a connection with db,  "sqlite:///" => connection format string

In [ ]:
engine = create_engine(r'sqlite:///D:\Sandeep Kaur\Projects\Youtube Analysis\YouTube_DataAnalysis_Project\export_data/youtube_data.sqlite')

In [ ]:
full_df[0:5000].to_sql('users', con = engine, if_exists = "replace")

# EDA, Category Performance Analysis -> which category dominates YOUTUBE?

In [ ]:
# we have to show the trending momentum for to 6 categories and for that check the data type of treanding_date 

In [ ]:
full_df.dtypes

In [ ]:
full_df["trending_date"]

In [ ]:
## to convert this data type from str to date-time, we will call pd.to_datetime()

In [ ]:
full_df["trending_date"] = pd.to_datetime(full_df["trending_date"], format = "%y.%d.%m")

In [ ]:
full_df["trending_date"]

In [ ]:
full_df.dtypes

In [ ]:
### here w  from json file

In [ ]:
import json

In [ ]:
path = r"D:\Sandeep Kaur\Projects\Youtube Analysis\YouTube_DataAnalysis_Project\additional_data\US_category_id.json"

In [ ]:
with open(path, 'r', encoding = "utf-8") as f:
    data = json.load(f)


In [ ]:
data

In [ ]:
data['items'][0]

In [ ]:
data['items'][0]['snippet']['title']

In [ ]:
data['items'][0]['id']

In [ ]:
cat_dict = {}
for item in data['items']:
    cat_dict[int(item['id'])] = item['snippet']['title']

In [ ]:
cat_dict

In [ ]:
full_df["category_id"].head(10)

In [ ]:
full_df["category_name"] = full_df["category_id"].map(cat_dict)

In [ ]:
full_df["category_name"]

In [ ]:
full_df.head(3)

In [ ]:
full_df.groupby(['trending_date', 'category_name'])['views'].sum()

In [ ]:
###    .unstack() ==>   unstack or reshape this data into a pivot

In [ ]:
full_df.groupby(['trending_date', 'category_name'])['views'].sum().unstack()

In [ ]:
###   fill_value = 0      =====>    to fill missing or Nan vlaues with 00

In [ ]:
pivot_df = full_df.groupby(['trending_date', 'category_name'])['views'].sum().unstack(fill_value = 0)

In [ ]:
pivot_df

In [ ]:
##    calling area() from plotly -- 

In [ ]:
area_chart = px.area(
    data_frame = pivot_df,
    x = pivot_df.index,
    y = pivot_df.columns, 
    title = "Trending Momentum over Time by Category"
)

In [ ]:
area_chart

In [ ]:
top_categories = full_df.groupby(['category_name'])['views'].sum().nlargest(6).index

In [ ]:
top_categories

In [ ]:
filtered_df = pivot_df[top_categories]

In [ ]:
filtered_df

In [ ]:
area_chart = px.area(
    data_frame = filtered_df,
    x = filtered_df.index,
    y = filtered_df.columns, 
    title = "Trending Momentum over Time by Category"
)

In [ ]:
area_chart

In [ ]:
####   TAKE-Aways ====> Area chart for to-6 most viewed categories - Music consistenly lead overall attention, but its dominace keep on fluctuating with time, and same for entertainment and sorts as well, They drive some short-term spikes reflecting the event based interest, 

# Views and Engagement Analysis

In [ ]:
#7.  DO VIRAL VIDEOS ACTUALLY GET ENGAGEMENT OR NOT?

In [ ]:
full_df.columns

In [ ]:
### create engagement column, cuz we donot have here in these columns, so ==>  engagement rate = (like + comment count) / all views

In [ ]:
full_df['engagement_rate'] = (full_df['likes'] + full_df['comment_count']) / full_df['views']

In [ ]:
full_df['engagement_rate']

In [ ]:
#### call scatter() from plotly package

In [ ]:
bubble_sample = full_df.sample(50000)

In [ ]:
bubble = px.scatter(bubble_sample,
    x='views',
    y='engagement_rate',
    color='category_name',
    size='comment_count',
    hover_name='title',
    title='Engagement Bubble Map: Views vs Engagement Rate',
    size_max=60
)

In [ ]:
bubble

In [ ]:
bubble.update_xaxes(type = 'log')

In [ ]:
###  It shows engagement drops as the views grows.// The trend is that we have videos, with have some low-med views and they actually have the high engagement. // Simillarly there are videos having very high views but actually have the low engagement. // Music category dominates the view but not the engagement 

# VIEWS VS ENGAGEMENT : INSIDE YOUTUBE'S ALGORITHM

In [ ]:
full_df.columns

In [ ]:
category_metrics = full_df.groupby('category_name').agg(
                                total_views = ('views', 'sum'),
                                avg_engagement_efficiency = ('engagement_rate', 'mean'),
                                video_count = ('video_id','count')).reset_index()

In [ ]:
category_metrics

In [ ]:
category_metrics.columns

In [ ]:
tree_map = px.treemap(
    category_metrics,
    path=['category_name'],
    values='total_views',
    color='avg_engagement_efficiency',
    color_continuous_scale='Viridis',
    title = 'Category attention share with Engagement Efficiency Overlay',
    hover_data = {
        'total_views' : ':,.0f',
        'avg_engagement_efficiency' : ':,.3f',
        'video_count': True
    }
)

In [ ]:
tree_map

In [ ]:
##  IT shows MUSIC, Entertainment and comedy are come of the demanding categories at youtube because these have pretty decent engagement  ====> some of those categories have litteraly good engagement efficiencies

# IS THE AUDIENCE ACTUALLY ENGAGED

In [ ]:
##  In order to figure out wheather audience is engaged or not->we need to perform the descriptive analysis

In [ ]:
full_df.columns

In [ ]:
full_df['engagement_rate'].describe()

In [ ]:
category_engagement_stat = full_df.groupby('category_name')[ 'engagement_rate'].describe()

In [ ]:
category_engagement_stat.sort_values('mean', ascending = False)

In [ ]:
full_df.columns

In [ ]:
box = px.box(full_df,
    x='category_name',
    y='engagement_rate',
    color='category_name',
    title='Audience Engagement by Category'
)

In [ ]:
box

# Key Findings

In [ ]:
# Most categories have low median engagement rates.
# Entertainment, Gaming, People & Blogs, Comedy, and Science & Technology show the highest engagement potential with several high-performing outliers.
# Movies, Shows, and Trailers have the lowest and most consistent engagement.
# Engagement is right-skewed, indicating that a small number of viral videos drive exceptionally high engagement across categories.

# Business Recommendations

In [ ]:
#Focus more on Entertainment, Gaming, Comedy, People & Blogs, and Science & Technology, as these categories have the highest potential for strong audience engagement.
#Analyze the characteristics of high-performing (viral) videos to identify successful content strategies.
#Improve content quality and audience interaction in Movies, Shows, and Trailers, which consistently show lower engagement.
#Measure both views and engagement when evaluating content performance, as high views alone do not guarantee audience interaction.